# Data cleaning

In [135]:
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow

In [136]:
BASE_DIR = Path().resolve().parents[0]
file_path = BASE_DIR / "data/raw/ValeursFoncieres-2025.txt"

In [137]:
cols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "No voie",
    "Type de voie",
    "Code voie",
    "Voie",
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Section",
    "No plan",
    "Code type local",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Surface terrain",
    "Nature culture",
    "Nature culture speciale"
]

dtype_map = {
    "Code postal": "str",
    "Code departement": "str",
    "Nature mutation": "category",
    "Type local": "category",
    "Type de voie": "category",
    "Commune": "category",
    "Nature culture": "str",
    "Nature culture speciale": "str",
}

In [138]:
df = pd.read_csv(file_path, sep="|", usecols=cols, dtype=dtype_map, parse_dates=["Date mutation"])

In [139]:
df.info(show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 3714829 entries, 0 to 3714828
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype   
---  ------                     --------------    -----   
 0   Date mutation              3714829 non-null  str     
 1   Nature mutation            3714829 non-null  category
 2   Valeur fonciere            3666409 non-null  str     
 3   No voie                    2400906 non-null  float64 
 4   Type de voie               2323331 non-null  category
 5   Code voie                  3694165 non-null  str     
 6   Voie                       3694160 non-null  str     
 7   Code postal                3694049 non-null  str     
 8   Commune                    3714829 non-null  category
 9   Code departement           3714829 non-null  str     
 10  Code commune               3714829 non-null  int64   
 11  Section                    3714623 non-null  str     
 12  No plan                    3714829 non-null  int64   
 13  Code typ

## Normalisation, conversion and parsing

In [140]:
# Category
df["Nature mutation"] = (
    df["Nature mutation"].astype(str).str.strip().astype("category")
)

df["Commune"] = (
    df["Commune"].astype(str).str.strip().astype("category")
)

df["Type local"] = (
    df["Type local"].astype(str).str.strip().astype("category")
)

# Int
df["Valeur fonciere"] = (
    df["Valeur fonciere"].astype("str").str.replace(",", ".", regex=False)
)
df["Valeur fonciere"] = pd.to_numeric(df["Valeur fonciere"], errors="coerce")

# Str
df["Type de voie"] = (
    df["Type de voie"].astype(str).str.strip().astype("str")
)

df["Code voie"] = (
    df["Code voie"].astype(str).str.strip().astype("str")
)

df["Voie"] = (
    df["Voie"].astype(str).str.strip().astype("str")
)

df["Code postal"] = (
    df["Code postal"].astype(str).str.strip().astype("str")
)

df["Code departement"] = (
    df["Code departement"].astype(str).str.strip().astype("str")
)

df["Section"] = (
    df["Section"].astype(str).str.strip().astype("str")
)

df["Nature culture"] = (
    df["Nature culture"].astype(str).str.strip().astype("str")
)

df["Nature culture speciale"] = (
    df["Nature culture speciale"].astype(str).str.strip().astype("str")
)

# Date
df["Date mutation"] = pd.to_datetime(df["Date mutation"], format="%d/%m/%Y")


## NA values

In [141]:
df = df.dropna(subset=["Valeur fonciere"])

In [142]:
df = df.dropna(subset=["Surface reelle bati"])

In [143]:
df = df[df["Surface reelle bati"] != 0]

In [144]:
df = df.dropna(subset=["Code postal"])

In [145]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 1230775 entries, 2 to 3714827
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Date mutation              1230775 non-null  datetime64[us]
 1   Nature mutation            1230775 non-null  category      
 2   Valeur fonciere            1230775 non-null  float64       
 3   No voie                    1223753 non-null  float64       
 4   Type de voie               1115868 non-null  str           
 5   Code voie                  1230775 non-null  str           
 6   Voie                       1230774 non-null  str           
 7   Code postal                1230775 non-null  str           
 8   Commune                    1230775 non-null  category      
 9   Code departement           1230775 non-null  str           
 10  Code commune               1230775 non-null  int64         
 11  Section                    1230723 non-null  str     

## Use of category type

In [146]:
cat_cols_used = []
cat_cols = df.select_dtypes(include="category").columns

for col in cat_cols:
    nb_used = df[col].nunique()
    nb_total = len(df[col].cat.categories)

    cat_cols_used.append({
        "column": col,
        "used": nb_used,
        "unused": nb_total - nb_used,
        "total": nb_total
    })

report_df = pd.DataFrame(cat_cols_used)
report_df.sort_values("unused", ascending=False)
print(report_df)

            column   used  unused  total
0  Nature mutation      6       0      6
1          Commune  29131    1645  30776
2       Type local      3       1      4


### Unused category in "`Total local`"

'Dépendance' value is unused (count = 0)

In [147]:
df["Type local"].value_counts()

Type local
Maison                                      617328
Appartement                                 505017
Local industriel. commercial ou assimilé    108430
Dépendance                                       0
Name: count, dtype: int64

### Clean category (unused)

In [148]:
df["Type local"] = df["Type local"].cat.remove_unused_categories()

In [149]:
df["Commune"] = df["Commune"].cat.remove_unused_categories()

## Filter on "`Nature culture`" and "`Nature culture speciale`"

In [150]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 1230775 entries, 2 to 3714827
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Date mutation              1230775 non-null  datetime64[us]
 1   Nature mutation            1230775 non-null  category      
 2   Valeur fonciere            1230775 non-null  float64       
 3   No voie                    1223753 non-null  float64       
 4   Type de voie               1115868 non-null  str           
 5   Code voie                  1230775 non-null  str           
 6   Voie                       1230774 non-null  str           
 7   Code postal                1230775 non-null  str           
 8   Commune                    1230775 non-null  category      
 9   Code departement           1230775 non-null  str           
 10  Code commune               1230775 non-null  int64         
 11  Section                    1230723 non-null  str     

In [151]:
df = df[
    df["Nature culture"].isna() |
    (df["Nature culture"].astype(str).str.strip() == "")
]

In [152]:
df = df[
    df["Nature culture speciale"].isna() |
    (df["Nature culture speciale"].astype(str).str.strip() == "")
]

In [153]:
df.drop(labels=["Nature culture", "Nature culture speciale", "Surface terrain"], axis="columns", inplace=True)

In [154]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 474711 entries, 18 to 3714827
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Date mutation              474711 non-null  datetime64[us]
 1   Nature mutation            474711 non-null  category      
 2   Valeur fonciere            474711 non-null  float64       
 3   No voie                    470718 non-null  float64       
 4   Type de voie               459403 non-null  str           
 5   Code voie                  474711 non-null  str           
 6   Voie                       474710 non-null  str           
 7   Code postal                474711 non-null  str           
 8   Commune                    474711 non-null  category      
 9   Code departement           474711 non-null  str           
 10  Code commune               474711 non-null  int64         
 11  Section                    474685 non-null  str           
 12  No

## Handling anomalies

In [155]:
df = df[
    df["Valeur fonciere"].notna() &
    (df["Valeur fonciere"] >= 1)
]

df = df[
    df["Surface reelle bati"].notna() &
    (df["Surface reelle bati"] >= 1)
]

## Add new data

In [156]:
df["prix_m2"] = df["Valeur fonciere"] / df["Surface reelle bati"]

In [157]:
df["prix_m2"] = df["prix_m2"].replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(subset=["prix_m2"])

In [158]:
df["prix_m2"].describe()

count    4.747100e+05
mean     1.797351e+04
std      1.731262e+05
min      2.622057e-05
25%      2.185714e+03
50%      3.442857e+03
75%      5.571429e+03
max      3.391250e+07
Name: prix_m2, dtype: float64

In [159]:
df = df[
    (df["prix_m2"] > df["prix_m2"].quantile(0.01)) &
    (df["prix_m2"] < df["prix_m2"].quantile(0.99))
]

In [160]:
df["prix_m2"].describe()

count    465206.000000
mean       6840.525667
std       18489.588594
min         161.538462
25%        2210.526316
50%        3442.857143
75%        5500.000000
max      324871.965986
Name: prix_m2, dtype: float64

## Final check

In [161]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 465206 entries, 18 to 3714827
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Date mutation              465206 non-null  datetime64[us]
 1   Nature mutation            465206 non-null  category      
 2   Valeur fonciere            465206 non-null  float64       
 3   No voie                    461241 non-null  float64       
 4   Type de voie               450313 non-null  str           
 5   Code voie                  465206 non-null  str           
 6   Voie                       465205 non-null  str           
 7   Code postal                465206 non-null  str           
 8   Commune                    465206 non-null  category      
 9   Code departement           465206 non-null  str           
 10  Code commune               465206 non-null  int64         
 11  Section                    465180 non-null  str           
 12  No

In [162]:
df.isna().sum().sort_values(ascending=False).head(20)

Type de voie                 14893
No voie                       3965
Section                         26
Voie                             1
Nature mutation                  0
Date mutation                    0
Code voie                        0
Code postal                      0
Commune                          0
Valeur fonciere                  0
Code departement                 0
Code commune                     0
No plan                          0
Code type local                  0
Type local                       0
Surface reelle bati              0
Nombre pieces principales        0
prix_m2                          0
dtype: int64

In [163]:
print("Length :", len(df))
print("Prix/m² médian :", df["prix_m2"].median())
print("Communes :", df["Commune"].nunique())

Length : 465206
Prix/m² médian : 3442.8571428571427
Communes : 9490


## Export data

In [164]:
clean_dir = BASE_DIR / "data/clean"
clean_dir.mkdir(exist_ok=True)

df.to_parquet(
    clean_dir / "dvf_clean.parquet",
    index=False
)